# kaNAIRO — free Kaggle image-to-3D asset lab

Project rules: read `skills/kaggle-asset-lab/SKILL.md`, `skills/blender-nairobi-game-art/SKILL.md`, and `docs/BLENDER_ASSETS.md` in the cloned game repository before preparing a model for the game.

TripoSR reconstructs a 3D guess from an image; it does not make assets from text, promise correct unseen sides, create game animation, or guarantee production-ready topology. Start with one clean reference and review the result in Blender before integrating it. Only upload images you have the right to use.



In [ ]:
from pathlib import Path
import subprocess, sys, json, os

assert Path('/kaggle').exists(), 'Run this notebook on Kaggle.'
subprocess.run(['nvidia-smi'], check=False)
WORK = Path('/kaggle/working/nairoblife-assets')
WORK.mkdir(parents=True, exist_ok=True)
print('Outputs will be written to:', WORK)

## Clone the project instructions and TripoSR

The game clone is only for its project-specific skill, asset rules and provenance notes. TripoSR code and pretrained checkpoint are MIT-licensed according to its upstream README. Check the upstream license and current dependency instructions before each new run.

In [ ]:
GAME = Path('/kaggle/working/NairobLife')
TRIPOSR = Path('/kaggle/working/TripoSR')
if not GAME.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'arena/f8aec938-nairoblife', '--single-branch', 'https://github.com/wambetebenjamin/NairobLife.git', str(GAME)], check=True)
if not TRIPOSR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/VAST-AI-Research/TripoSR.git', str(TRIPOSR)], check=True)
print((GAME / 'skills/kaggle-asset-lab/SKILL.md').read_text())
print((GAME / 'skills/blender-nairobi-game-art/SKILL.md').read_text()[:5000])
print('TripoSR checkout:', subprocess.check_output(['git','-C',str(TRIPOSR),'rev-parse','HEAD'], text=True).strip())

## Install TripoSR dependencies

Kaggle already provides PyTorch in many GPU images. The upstream requirements file intentionally does not pin/install PyTorch. Dependency resolution or CUDA extension compilation can still fail when Kaggle's image changes; stop and inspect the error rather than blindly upgrading the preinstalled CUDA/PyTorch stack.

In [ ]:
%cd /kaggle/working/TripoSR
!python -m pip install -q --upgrade setuptools
!python -m pip install -q -r requirements.txt
!python -m pip install -q git+https://github.com/tatsy/torchmcubes.git
import torch
print('torch:', torch.__version__, 'CUDA:', torch.version.cuda, 'available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator before running model inference.')

## Select one reference image

Upload a licensed image as a Kaggle dataset/input, then set `IMAGE` below to its mounted path. Prefer a single, isolated object on a plain background; whole scenes and multiple objects produce poor game meshes. For a car, use a clear three-quarter view; for a standing person, use a full-body view with limbs visible. The model still infers unseen sides from that one view.

In [ ]:
from pathlib import Path
from PIL import Image

inputs = list(Path('/kaggle/input').rglob('*')) if Path('/kaggle/input').exists() else []
images = [p for p in inputs if p.suffix.lower() in {'.png','.jpg','.jpeg','.webp'}]
print('Images visible to this notebook:')
for p in images[:40]: print(p)
# CHANGE this to one image listed above, from a source you are allowed to use.
IMAGE = Path('/kaggle/input/your-reference-dataset/your-reference.png')
if not IMAGE.is_file():
    raise FileNotFoundError('Set IMAGE to a real image path under /kaggle/input, then rerun this cell.')
display(Image.open(IMAGE).convert('RGB').resize((512,512)))

## Generate one GLB candidate

This asks TripoSR for a 1K baked texture and binary glTF output. If texture baking runs out of memory, retry once without `--bake-texture`; the default output uses vertex colors. Do not bulk-generate until the first model has been inspected.

In [ ]:
import datetime, shlex
ASSET_NAME = 'candidate'  # e.g. 'nairobi-hatchback' or 'pedestrian-rosa'
OUT = WORK / ASSET_NAME
OUT.mkdir(parents=True, exist_ok=True)
cmd = [sys.executable, 'run.py', str(IMAGE), '--device', 'cuda:0', '--model-save-format', 'glb', '--bake-texture', '--texture-resolution', '1024', '--output-dir', str(OUT)]
print('Running:', shlex.join(cmd))
subprocess.run(cmd, cwd=TRIPOSR, check=True)
glbs = sorted(OUT.rglob('*.glb'))
print('GLB outputs:', glbs)
assert glbs, 'No GLB output found; inspect the inference log and TripoSR docs.'
manifest = {'asset_name': ASSET_NAME, 'created_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'source_image_name': IMAGE.name, 'source_repo': 'https://github.com/VAST-AI-Research/TripoSR', 'source_commit': subprocess.check_output(['git','-C',str(TRIPOSR),'rev-parse','HEAD'], text=True).strip(), 'model': 'stabilityai/TripoSR', 'license_note': 'TripoSR README states code and pretrained model are MIT; verify current terms before distribution.', 'review_status': 'candidate only; inspect orientation, scale, topology, materials, rights and game performance before use'}
(OUT / 'asset-manifest.json').write_text(json.dumps(manifest, indent=2))
for p in glbs: print(p, p.stat().st_size, 'bytes')

## Review and bring back to the game

Kaggle output is not game-ready simply because it is GLB. Inspect in Blender: check dimensions in metres, ground contact/origin, Y-up and forward -Z, normals, topology, texture, transparency and visual quality. The game skill targets under 2 MB per hero GLB where practical and requires a Three.js fallback. TripoSR does not rig people or create animation clips, so do not use its output to replace the animated player/crowd without a separate rigging/animation pass.

Download only the reviewed `.glb` and `asset-manifest.json` from Kaggle outputs. Add it to `public/assets/` only after it has been integrated and tested in the actual game; update `docs/FREE_ASSETS.md` with provenance. Never copy Kaggle checkpoints, caches, arbitrary input photos, or tokens into Git.